# Barter-RS: Low-Level WebSocket Integration

This notebook demonstrates the `barter-integration` crate — the foundation layer
for building custom exchange integrations with WebSocket and REST protocols.

## Topics Covered
- The `Transformer` trait for stateful data processing
- `ExchangeStream` for consuming WebSocket data
- Building a custom integration from scratch (Binance aggTrade example)
- The `Validator` trait for input validation
- Core abstractions: `Unrecoverable`, `Terminal`

In [ ]:
:dep barter-integration = { version = "0.10", features = ["channel", "collection", "error", "protocol", "serde", "subscription", "stream"] }
:dep tokio = { version = "1", features = ["full"] }
:dep tokio-tungstenite = { version = "0.26", features = ["url", "rustls-tls-webpki-roots"] }
:dep futures = "0.3"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep tracing = "0.1"

## 1. Core Traits Overview

| Trait | Signature | Purpose |
|-------|-----------|--------|
| `Transformer` | `Input → Vec<Result<Output, Error>>` | Stateful data transformation pipeline |
| `Validator` | `&State → Result<(), Error>` | Validate conditions on state |
| `Unrecoverable` | `&self → bool` | Classify errors as fatal or transient |
| `Terminal` | `&self → bool` | Mark events that end a stream's lifecycle |

These traits compose to build the full data pipeline:
```
WebSocket bytes → Deserialise → Validate → Transform → Output Stream
```

## 2. The Transformer Trait

A `Transformer` converts parsed exchange messages into your desired output model.
It can hold state (e.g., running totals, sequence numbers, order book snapshots).

In [ ]:
use barter_integration::{Transformer, error::SocketError};
use serde::Deserialize;

/// Raw Binance message — could be a subscription confirmation or a trade
#[derive(Debug, Deserialize)]
#[serde(untagged)]
enum BinanceMessage {
    SubResponse {
        result: Option<Vec<String>>,
        id: u32,
    },
    Trade {
        #[serde(rename = "s")]
        symbol: String,
        #[serde(rename = "p")]
        price: String,
        #[serde(rename = "q")]
        quantity: String,
        #[serde(rename = "m")]
        is_buyer_maker: bool,
    },
}

/// Our desired output: a simplified trade summary
#[derive(Debug)]
struct TradeSummary {
    symbol: String,
    price: f64,
    quantity: f64,
    side: &'static str,
    trade_count: u64,
}

/// Stateful transformer that counts trades and converts raw messages
struct TradeTransformer {
    trade_count: u64,
}

impl Transformer for TradeTransformer {
    type Error = SocketError;
    type Input = BinanceMessage;
    type Output = TradeSummary;
    type OutputIter = Vec<Result<Self::Output, Self::Error>>;

    fn transform(&mut self, input: Self::Input) -> Self::OutputIter {
        match input {
            BinanceMessage::SubResponse { .. } => {
                // Subscription confirmations produce no output
                vec![]
            }
            BinanceMessage::Trade { symbol, price, quantity, is_buyer_maker } => {
                self.trade_count += 1;
                vec![Ok(TradeSummary {
                    symbol,
                    price: price.parse().unwrap_or(0.0),
                    quantity: quantity.parse().unwrap_or(0.0),
                    side: if is_buyer_maker { "Sell" } else { "Buy" },
                    trade_count: self.trade_count,
                })]
            }
        }
    }
}

println!("TradeTransformer defined.");
println!("  Input:  BinanceMessage (raw JSON)");
println!("  Output: TradeSummary (normalised)");
println!("  State:  trade_count (running counter)");

## 3. ExchangeStream — Putting It Together

An `ExchangeStream` wraps a WebSocket connection with a `Transformer` to produce
a standard `futures::Stream` of your output type. This is the core building block
used by `barter-data` for all exchange integrations.

```rust
type ExchangeWsStream<T> = ExchangeStream<WebSocketSerdeParser, WebSocket, T>;
```

In [ ]:
use barter_integration::{
    protocol::websocket::{WebSocket, WebSocketSerdeParser, WsMessage},
    stream::ExchangeStream,
};
use futures::{SinkExt, StreamExt};
use std::collections::VecDeque;
use tokio_tungstenite::connect_async;

// Type alias for a WebSocket-based ExchangeStream
type ExchangeWsStream<T> = ExchangeStream<WebSocketSerdeParser, WebSocket, T>;

// Connect to Binance WebSocket
let mut ws_conn = connect_async("wss://fstream.binance.com/ws")
    .await
    .map(|(ws, _)| ws)
    .expect("failed to connect to Binance");

// Subscribe to BTC/USDT aggregated trades
ws_conn
    .send(WsMessage::text(
        serde_json::json!({
            "method": "SUBSCRIBE",
            "params": ["btcusdt@aggTrade"],
            "id": 1
        }).to_string()
    ))
    .await
    .expect("failed to subscribe");

// Create the ExchangeStream with our transformer
let transformer = TradeTransformer { trade_count: 0 };
let mut stream = ExchangeWsStream::new(ws_conn, transformer, VecDeque::new());

println!("Connected! Reading first 5 transformed trades...\n");

let mut count = 0;
while let Some(result) = stream.next().await {
    match result {
        Ok(summary) => {
            println!(
                "  #{} {} {} {:.2} @ {:.2}",
                summary.trade_count, summary.symbol,
                summary.side, summary.quantity, summary.price
            );
            count += 1;
            if count >= 5 { break; }
        }
        Err(e) => eprintln!("  Error: {e:?}"),
    }
}

println!("\nDone! Transformer processed {count} trades with running state.");

## 4. Validator Trait

Validators check conditions on data before it enters the transformation pipeline.
This is used internally by exchange integrations to validate subscription
responses, sequence numbers, and message integrity.

In [ ]:
use barter_integration::Validator;

/// Example: validate that a trade has positive quantity
#[derive(Debug)]
struct PositiveQuantity;

impl Validator for PositiveQuantity {
    type Context = f64; // the quantity to validate
    type Error = String;

    fn validate(&self, quantity: &Self::Context) -> Result<(), Self::Error> {
        if *quantity > 0.0 {
            Ok(())
        } else {
            Err(format!("Invalid quantity: {quantity}"))
        }
    }
}

let validator = PositiveQuantity;
assert!(validator.validate(&1.5).is_ok());
assert!(validator.validate(&-0.1).is_err());

println!("Validator example: PositiveQuantity");
println!("  validate(1.5)  → Ok");
println!("  validate(-0.1) → Err");

## 5. Error Classification: Unrecoverable & Terminal

Barter distinguishes between transient and fatal errors at the type level:

- **`Unrecoverable`**: Is this error fatal? Should we shut down?
- **`Terminal`**: Has the stream ended? Should we reconnect?

This drives the reconnection and shutdown logic in the engine.

In [ ]:
use barter_integration::{Unrecoverable, Terminal};

// Example error that can be either recoverable or not
#[derive(Debug)]
enum StreamError {
    /// Transient: can retry
    Timeout,
    /// Fatal: must stop
    AuthFailed,
    /// Stream ended naturally
    Disconnected,
}

impl Unrecoverable for StreamError {
    fn is_unrecoverable(&self) -> bool {
        matches!(self, StreamError::AuthFailed)
    }
}

impl Terminal for StreamError {
    fn is_terminal(&self) -> bool {
        matches!(self, StreamError::Disconnected | StreamError::AuthFailed)
    }
}

let errors = [StreamError::Timeout, StreamError::AuthFailed, StreamError::Disconnected];
for err in &errors {
    println!(
        "  {:?}: unrecoverable={}, terminal={}",
        err, err.is_unrecoverable(), err.is_terminal()
    );
}

## Integration Architecture

```
                          barter-integration
┌──────────────────────────────────────────────────────────┐
│                                                          │
│  WebSocket / REST                                        │
│       │                                                  │
│       ▼                                                  │
│  ┌──────────┐    ┌───────────┐    ┌──────────────┐      │
│  │  Parser   │───▶│ Validator  │───▶│ Transformer   │     │
│  │ (deser)   │    │ (check)   │    │ (stateful)    │     │
│  └──────────┘    └───────────┘    └──────┬────────┘     │
│                                          │              │
│                                          ▼              │
│                                   ExchangeStream        │
│                                  (futures::Stream)      │
└──────────────────────────────────────────────────────────┘
                         │
                         ▼
                    barter-data
              (exchange implementations)
```

### When to Use barter-integration Directly

- Adding support for a **new exchange** not yet in barter-data
- Building a **custom data pipeline** outside the Barter engine
- Implementing **private WebSocket feeds** (account updates, order status)
- Creating **REST API clients** for order management